# Trajectory QA (human vs synthetic)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# notebooks/ → project root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import RNG_SEED
print("ROOT =", ROOT)


## Knobs

In [ ]:
N_BOTS = None       # None = one bot per human game/session
N_PREVIEW = 5       # trajectories kept for plots / single-trace sanity
RUN_RE = True
RUN_LOL = True
RUN_CSGO = True

VAE_FORCE_RETRAIN = False


## 1) Load humans + feature tables

In [ ]:
from src.data import find_red_eclipse_files, load_red_eclipse_mouse
from src.features import extract_features

games_df = None
re_human_mice = None

if RUN_RE:
    game_files = find_red_eclipse_files()
    game_rows, re_human_mice = [], []
    for file_path in game_files:
        meta, mouse = load_red_eclipse_mouse(file_path)
        feats = extract_features(mouse)
        if feats is None:
            continue
        feats.update({**meta, "is_bot": 0, "bot_type": "human"})
        game_rows.append(feats)
        re_human_mice.append(mouse)
    games_df = pd.DataFrame(game_rows)
    print(f"RE humans: {len(games_df)} games, {games_df['userId'].nunique()} users")
else:
    print("Skip RE")


In [ ]:
from src.data import load_lol_match_windows
from src.config import (
    LOL_MATCH_WINDOW_START_MIN,
    LOL_MATCH_WINDOW_END_MIN,
)

lol_games_df = None
lol_human_mice = None

if RUN_LOL:
    lol_records, lol_load_summary = load_lol_match_windows()
    print("LoL match-window load:", lol_load_summary)
    lol_rows, lol_human_mice = [], []
    for rec in lol_records:
        mouse = rec["mouse"]
        feats = extract_features(mouse)
        if feats is None:
            continue
        feats.update({
            "userId": rec["userId"],
            "gameId": rec["gameId"],
            "source_file": rec["source_file"],
            "is_bot": 0,
            "bot_type": "human",
        })
        lol_rows.append(feats)
        lol_human_mice.append(mouse)
    lol_games_df = pd.DataFrame(lol_rows)
    print(
        f"LoL humans: {len(lol_games_df)} windows, "
        f"{lol_games_df['userId'].nunique()} users "
        f"({LOL_MATCH_WINDOW_START_MIN}–{LOL_MATCH_WINDOW_END_MIN} min)"
    )
else:
    print("Skip LoL")


In [ ]:
from pathlib import Path
from src.config import CSGO_DATA_ROOT, CSGO_WINDOW_MIN
from src.data_csgo import (
    check_axis_convention,
    eye_vectors_to_mouse_df,
    validate_real_roundtrip,
    window_mouse_round_alive,
)

csgo_games_df = None
csgo_mouse_win = None

if RUN_CSGO:
    USECOLS = ["time", "eyeVectorX", "eyeVectorY", "eyeVectorZ"]

    def find_gameflt_files(root=CSGO_DATA_ROOT):
        files = sorted(Path(root).rglob("gameFlt.csv"))
        print(f"Found {len(files)} gameFlt.csv under {root}")
        return files

    def load_eye_csv(path):
        df = pd.read_csv(path, usecols=USECOLS)
        finite = np.isfinite(df[["eyeVectorX", "eyeVectorY", "eyeVectorZ"]]).all(axis=1)
        return df.loc[finite].reset_index(drop=True)

    def convert_one(path):
        flt = load_eye_csv(path)
        mouse_df, meta = eye_vectors_to_mouse_df(
            flt["time"], flt["eyeVectorX"], flt["eyeVectorY"], flt["eyeVectorZ"]
        )
        return mouse_df, meta, flt

    gameflt_paths = find_gameflt_files()
    assert gameflt_paths, f"No gameFlt.csv under {CSGO_DATA_ROOT}"

    mouse0, meta0, flt0 = convert_one(gameflt_paths[0])
    axis_ok = check_axis_convention(flt0["eyeVectorY"])
    rt = validate_real_roundtrip(
        flt0["eyeVectorX"], flt0["eyeVectorY"], flt0["eyeVectorZ"]
    )
    if not (axis_ok and rt["ok"]):
        raise RuntimeError(
            f"CSGO first-file checks failed: axis_ok={axis_ok}, roundtrip_ok={rt['ok']}"
        )

    csgo_mouse = {}
    for path in gameflt_paths:
        participant = path.parent.name
        session = path.parent.parent.name
        try:
            mouse_df, meta, _ = convert_one(path)
        except Exception as e:
            print(f"  SKIP {(session, participant)}: {e}")
            continue
        csgo_mouse[(session, participant)] = mouse_df

    csgo_mouse_win = {}
    csgo_rows = []
    n_skip = 0
    for (session, participant), mouse in csgo_mouse.items():
        session_dir = Path(CSGO_DATA_ROOT) / session / participant
        win, meta = window_mouse_round_alive(
            mouse, session_dir, window_min=CSGO_WINDOW_MIN
        )
        if not meta.get("ok") or win is None:
            n_skip += 1
            continue
        feats = extract_features(win)
        if feats is None:
            n_skip += 1
            continue
        csgo_mouse_win[(session, participant)] = win
        feats.update({
            "userId": f"csgo_{session}_{participant}",
            "gameId": f"{session}_{participant}",
            "session": session,
            "participant": participant,
            "is_bot": 0,
            "bot_type": "human",
        })
        csgo_rows.append(feats)
    csgo_games_df = pd.DataFrame(csgo_rows)
    print(
        f"CSGO humans: {len(csgo_games_df)} Round2+alive sessions "
        f"(skipped {n_skip})"
    )
else:
    print("Skip CSGO")


## 2) Generate bots (duration-aligned; load VAE weights)

In [ ]:
from src.bots import (
    build_segments,
    collect_human_motion_samples,
    generate_bot_mouse_games,
    estimate_smooth_params,
    estimate_bezier_params,
)
from src.features import extract_features
from src.vae_bot import (
    ensure_vae_bundle,
    DEFAULT_RE_WEIGHTS,
    DEFAULT_LOL_WEIGHTS,
    DEFAULT_CSGO_WEIGHTS,
)


def _feat_table(mice, ids, user_id, bot_type):
    rows = []
    for mouse, gid in zip(mice, ids):
        feats = extract_features(mouse)
        if feats is None:
            continue
        feats.update({
            "userId": user_id,
            "gameId": gid,
            "is_bot": 1,
            "bot_type": bot_type,
        })
        rows.append(feats)
    return pd.DataFrame(rows)


def prepare_game(name, human_mice, human_feat, *, round_deltas, vae_path, rng_seed, id_prefix):
    mice = list(human_mice)
    n_bots = N_BOTS if N_BOTS is not None else len(human_feat)
    motion = collect_human_motion_samples(mice, rng=np.random.default_rng(rng_seed))
    step_median = float(np.median(motion["step_samples"]))
    vae_bundle = ensure_vae_bundle(
        mice,
        step_median,
        path=vae_path,
        force_retrain=VAE_FORCE_RETRAIN,
        seed=rng_seed,
    )
    games = generate_bot_mouse_games(
        mice,
        human_feat,
        n_bots=n_bots,
        rng_seed=rng_seed,
        round_deltas=round_deltas,
        id_prefix=id_prefix,
        vae_bundle=vae_bundle,
    )
    rng = np.random.default_rng(rng_seed)
    segment_pool = []
    for m in mice:
        segment_pool.extend(build_segments(m, rng=rng))

    bot_feats = {
        "stitch": _feat_table(games["stitch"], games["stitch_ids"], -1, "stitch"),
        "smooth": _feat_table(games["smooth"], games["smooth_ids"], -2, "smooth"),
        "bezier": _feat_table(games["bezier"], games["bezier_ids"], -3, "bezier"),
        "vae": _feat_table(games["vae"], games["vae_ids"], -4, "vae"),
    }
    samples = {
        "human": mice,
        "stitch": games["stitch"][:N_PREVIEW],
        "smooth": games["smooth"][:N_PREVIEW],
        "bezier": games["bezier"][:N_PREVIEW],
        "vae": games["vae"][:N_PREVIEW],
    }
    print(
        f"{name}: n_bots={n_bots} target={games['target_ms']/1000:.1f}s "
        f"segments={len(segment_pool)} vae={Path(vae_path).name} "
        f"norm={vae_bundle.get('norm')} axis_scale={vae_bundle.get('axis_scale')}"
    )
    return {
        "name": name,
        "human_feat": human_feat,
        "bot_feats": bot_feats,
        "human_mice": mice,
        "samples": samples,
        "segment_pool": segment_pool,
        "dt_samples": motion["dt_samples"],
        "dt_by_session": motion["dt_by_session"],
        "target_ms": games["target_ms"],
        "smooth_params": estimate_smooth_params(human_feat, **motion),
        "bezier_params": estimate_bezier_params(human_feat, **motion),
        "vae_bundle": vae_bundle,
    }


GAME_SPECS = []
if RUN_RE and games_df is not None:
    GAME_SPECS.append(
        prepare_game(
            "RE",
            re_human_mice,
            games_df,
            round_deltas=True,
            vae_path=DEFAULT_RE_WEIGHTS,
            rng_seed=RNG_SEED,
            id_prefix="re",
        )
    )
if RUN_LOL and lol_games_df is not None:
    GAME_SPECS.append(
        prepare_game(
            "LoL",
            lol_human_mice,
            lol_games_df,
            round_deltas=True,
            vae_path=DEFAULT_LOL_WEIGHTS,
            rng_seed=RNG_SEED + 1,
            id_prefix="lol",
        )
    )
if RUN_CSGO and csgo_games_df is not None:
    GAME_SPECS.append(
        prepare_game(
            "CSGO",
            list(csgo_mouse_win.values()),
            csgo_games_df,
            round_deltas=False,
            vae_path=DEFAULT_CSGO_WEIGHTS,
            rng_seed=RNG_SEED + 2,
            id_prefix="csgo",
        )
    )

print("Prepared games:", [g["name"] for g in GAME_SPECS])
assert GAME_SPECS, "No games prepared — enable RUN_* and ensure data paths exist"


## 3) Trajectory QA

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.features import extract_features, feature_cols
from src.bots import (
    smooth_params_for_print,
    bezier_params_for_print,
)
from src.config import RNG_SEED

try:
    from src.vae_bot import sample_vae_segments
except Exception:
    sample_vae_segments = None

QA_COLS = [
    c for c in [
        "n_events", "mean_dt", "avg_speed", "speed_max", "speed_std",
        "idle_ratio", "turn_angle_mean", "dist_std", "total_movement",
    ]
    if c in feature_cols or c in ("n_events",)
]
if "n_events" not in QA_COLS:
    QA_COLS = ["n_events"] + QA_COLS


def _first_mouse(xs):
    if xs is None:
        return None
    if isinstance(xs, pd.DataFrame):
        return xs
    if isinstance(xs, (list, tuple)) and len(xs) and isinstance(xs[0], pd.DataFrame):
        return xs[0]
    return None


def _duration_ms(mouse):
    if mouse is None or len(mouse) < 2:
        return float("nan")
    return float(mouse["time"].iloc[-1] - mouse["time"].iloc[0])


def _sanity(mouse, label, target_ms=None):
    if mouse is None or len(mouse) == 0:
        print(f"  [{label}] MISSING / empty")
        return None
    t = mouse["time"].to_numpy(dtype=float)
    dx = mouse["dx"].to_numpy(dtype=float)
    dy = mouse["dy"].to_numpy(dtype=float)
    dt = np.diff(t)
    mono = bool(np.all(dt >= 0))
    n_bad_dt = int(np.sum(dt < 0))
    n_zero_dt = int(np.sum(dt == 0))
    dur = _duration_ms(mouse)
    dist = np.sqrt(dx[1:] ** 2 + dy[1:] ** 2)
    valid = dt > 0
    speed = dist[valid] / dt[valid] if np.any(valid) else np.array([])
    info = {
        "label": label,
        "len": len(mouse),
        "duration_ms": dur,
        "time_monotonic": mono,
        "n_time_reversals": n_bad_dt,
        "n_zero_dt": n_zero_dt,
        "n_nan": int(mouse[["dx", "dy", "time"]].isna().sum().sum()),
        "dx_p99": float(np.nanpercentile(np.abs(dx), 99)),
        "dy_p99": float(np.nanpercentile(np.abs(dy), 99)),
        "speed_max_seq": float(np.nanmax(speed)) if len(speed) else float("nan"),
    }
    if target_ms is not None:
        info["target_ms"] = float(target_ms)
        info["duration_vs_target"] = dur / float(target_ms) if target_ms else float("nan")
    flags = []
    if not mono:
        flags.append(f"TIME_REVERSAL x{n_bad_dt}")
    if n_zero_dt:
        flags.append(f"ZERO_DT x{n_zero_dt}")
    if info["n_nan"]:
        flags.append(f"NaN x{info['n_nan']}")
    if target_ms and np.isfinite(info.get("duration_vs_target", np.nan)):
        r = info["duration_vs_target"]
        if r < 0.5 or r > 1.5:
            flags.append(f"DURATION_OFF target_ratio={r:.2f}")
    if len(speed) and info["speed_max_seq"] > 500:
        flags.append(f"SPEED_BURST max={info['speed_max_seq']:.1f}")
    print(
        f"  [{label}] n={info['len']} dur={dur/1000:.1f}s "
        f"mono={mono} nan={info['n_nan']} "
        f"|dx|p99={info['dx_p99']:.2f} |dy|p99={info['dy_p99']:.2f} "
        f"speed_max={info['speed_max_seq']:.1f} zero_dt={n_zero_dt}"
        + (f"  FLAGS={flags}" if flags else "  OK")
    )
    return info


def _feat_row(mouse):
    feats = extract_features(mouse)
    if feats is None:
        return None
    return {k: feats.get(k, np.nan) for k in QA_COLS}


def _median_table(named_mice):
    cols_out = {}
    for name, obj in named_mice:
        if obj is None:
            continue
        if isinstance(obj, pd.DataFrame) and "avg_speed" in obj.columns:
            cols_out[name] = obj[QA_COLS].median(numeric_only=True)
            continue
        mice = obj if isinstance(obj, (list, tuple)) else [obj]
        rows = []
        for m in mice:
            if isinstance(m, pd.DataFrame) and set(["dx", "dy", "time"]).issubset(m.columns):
                r = _feat_row(m)
                if r:
                    rows.append(r)
        if not rows:
            continue
        cols_out[name] = pd.DataFrame(rows)[QA_COLS].median()
    if not cols_out:
        return None
    tab = pd.DataFrame(cols_out)
    if "human" in tab.columns:
        hum = tab["human"].replace(0, np.nan)
        for c in tab.columns:
            if c == "human":
                continue
            tab[f"{c}/H"] = (tab[c] / hum).round(2)
    return tab.round(3)


def _shared_axis_plot(named_mice, title, max_n=5):
    items = [(n, _first_mouse(m)) for n, m in named_mice]
    items = [(n, m) for n, m in items if m is not None and len(m) >= 2]
    if not items:
        print(f"  (skip plot: {title})")
        return
    xs, ys, paths = [], [], []
    for n, m in items[:max_n]:
        x = m["dx"].cumsum().to_numpy()
        y = m["dy"].cumsum().to_numpy()
        paths.append((n, x, y))
        xs.append(x)
        ys.append(y)
    all_x = np.concatenate(xs)
    all_y = np.concatenate(ys)
    pad_x = 0.05 * (all_x.max() - all_x.min() + 1e-9)
    pad_y = 0.05 * (all_y.max() - all_y.min() + 1e-9)
    xlim = (all_x.min() - pad_x, all_x.max() + pad_x)
    ylim = (all_y.min() - pad_y, all_y.max() + pad_y)

    n = len(paths)
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.2))
    if n == 1:
        axes = [axes]
    for ax, (name, x, y) in zip(axes, paths):
        ax.plot(x, y, lw=0.6)
        ax.set_title(name, fontsize=10)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_aspect("equal", adjustable="box")
        ax.tick_params(labelsize=7)
    fig.suptitle(title + " (shared axis limits)", fontsize=11)
    plt.tight_layout()
    plt.show()


def _flag_median_gaps(tab, human_col="human"):
    if tab is None or human_col not in tab.columns:
        return
    print("  Red flags (|log10 ratio| >= 1 vs human, i.e. ≥10× or ≤0.1×):")
    any_flag = False
    for col in tab.columns:
        if col == human_col or col.endswith("/H"):
            continue
        ratio_col = f"{col}/H"
        if ratio_col not in tab.columns:
            continue
        for feat in tab.index:
            r = tab.loc[feat, ratio_col]
            if pd.isna(r) or r <= 0:
                continue
            if abs(np.log10(float(r))) >= 1.0:
                print(f"    {col:10s}  {feat:16s}  median_ratio={r}")
                any_flag = True
    if not any_flag:
        print("    (none)")


assert "GAME_SPECS" in globals() and GAME_SPECS, "Run the prepare cell first"
print(f"Trajectory QA | games={[g['name'] for g in GAME_SPECS]} | cols={QA_COLS}")

for spec in GAME_SPECS:
    name = spec["name"]
    print("\n" + "=" * 72)
    print(f"# {name}")
    print("=" * 72)

    print("\n## 1) Feature medians (table) + ratio vs human")
    named = [("human", spec["human_feat"])]
    for bt, df in spec["bot_feats"].items():
        if df is not None and len(df):
            named.append((bt, df))
    tab = _median_table(named)
    if tab is not None:
        val_cols = [c for c in tab.columns if not c.endswith("/H")]
        ratio_cols = [c for c in tab.columns if c.endswith("/H")]
        print(tab[val_cols].to_string())
        if ratio_cols:
            print("\n  ratio bot/human:")
            print(tab[ratio_cols].to_string())
        _flag_median_gaps(tab)
    else:
        print("  (no feature tables)")

    print("\n## 2) Single-trace sanity (first sample each)")
    target = spec["target_ms"]
    for label in ["human", "stitch", "smooth", "bezier", "vae"]:
        _sanity(_first_mouse(spec["samples"].get(label)), label, target_ms=target)

    print("\n## 3) Side-by-side paths (shared axis limits)")
    _shared_axis_plot(
        [(k, spec["samples"].get(k)) for k in ["human", "stitch", "smooth", "bezier", "vae"]],
        title=f"{name}: human vs bots",
    )

    print("\n## 4a) Stitch ablative: raw segment vs stitched game")
    pool = spec["segment_pool"]
    if pool and len(pool):
        seg = pool[0]
        if isinstance(seg, pd.DataFrame):
            seg_plot = pd.DataFrame({
                "dx": seg["dx"].to_numpy(),
                "dy": seg["dy"].to_numpy(),
                "time": seg["rel_time"].to_numpy() if "rel_time" in seg.columns else np.arange(len(seg)),
            })
        else:
            seg_plot = None
        stitched = _first_mouse(spec["samples"].get("stitch"))
        print(f"  segment_pool n={len(pool)} | first seg events={len(seg_plot) if seg_plot is not None else '?'}")
        _shared_axis_plot(
            [("raw_segment", seg_plot), ("stitched_game", stitched)],
            title=f"{name}: stitch raw segment vs full stitch",
        )
        if seg_plot is not None and stitched is not None:
            print("  sanity raw segment:")
            _sanity(seg_plot, "raw_segment", target_ms=None)
            print("  sanity stitched:")
            _sanity(stitched, "stitched_game", target_ms=target)
    else:
        print("  (no segment_pool)")

    print("\n## 4b) Smooth / Bézier estimated params")
    sp, bp = spec["smooth_params"], spec["bezier_params"]
    print("  smooth:", smooth_params_for_print(sp) if sp else "(missing)")
    print("  bezier:", bezier_params_for_print(bp) if bp else "(missing)")
    if spec["dt_samples"] is not None and len(spec["dt_samples"]):
        print(
            f"  human dt: n={len(spec['dt_samples'])} "
            f"median={np.median(spec['dt_samples']):.2f}ms "
            f"p90={np.percentile(spec['dt_samples'], 90):.2f}ms"
        )
    if target is not None:
        print(f"  target_duration_ms={target} ({target/1000:.1f}s)")

    print("\n## 4c) VAE ablative: single decoded segment vs full VAE game")
    bundle = spec["vae_bundle"]
    vae_game = _first_mouse(spec["samples"].get("vae"))
    if bundle is not None and sample_vae_segments is not None:
        segs = sample_vae_segments(
            bundle,
            n_segments=1,
            dt_samples=spec["dt_samples"],
            dt_by_session=spec["dt_by_session"],
            rng=np.random.default_rng(RNG_SEED + 99),
        )
        one = segs[0]
        vae_seg = pd.DataFrame({
            "dx": one["dx"].to_numpy(),
            "dy": one["dy"].to_numpy(),
            "time": one["rel_time"].to_numpy() if "rel_time" in one.columns else np.arange(len(one)),
        })
        print(f"  VAE single segment events={len(vae_seg)}")
        _sanity(vae_seg, "vae_segment", target_ms=None)
        _sanity(vae_game, "vae_full_game", target_ms=target)
        _shared_axis_plot(
            [("vae_segment", vae_seg), ("vae_full_game", vae_game)],
            title=f"{name}: VAE segment vs full game",
        )
    else:
        print("  (skip — need vae_bundle + sample_vae_segments)")

print("\n" + "=" * 72)
print("QA done. Paste the printed tables + flags (and note which plots look weird).")
print("=" * 72)
